# 03. Modeling

선정한 피처로 모델을 학습하고 검증 및 테스트 성능을 평가합니다.

## 00. 경로 설정 및 함수 불러오기

프로젝트 경로 등록 및 전처리, 피처 생성, 학습 함수 불러오기

In [1]:
from pathlib import Path
import sys
import importlib

import pandas as pd

# 프로젝트 경로
PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("프로젝트 폴더를 찾을 수 없습니다.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 전처리 함수
from src.preprocess import (
    BATCH_1_PATH,
    BATCH_2_PATH,
    load_mat,
    to_list_of_dicts,
    extract_summary,
    extract_cycle_qdlin,
)

# 피처 생성 함수
from src.features import build_feature_table

# 수정 모듈 갱신
from src import train as train_module
importlib.reload(train_module)

# 학습, 평가 함수
from src.train import (
    RANDOM_STATE,
    VALID_SIZE,
    CV_SPLITS,
    ALPHA_CANDIDATES,
    split_features_target,
    split_train_valid,
    build_pipeline,
    build_baseline,
    evaluate_baseline_cv,
    evaluate_train_cv,
    select_alpha,
    evaluate_model,
)

print("프로젝트 경로:", PROJECT_ROOT)
print("커널 실행 파일:", sys.executable)
print("random_state:", RANDOM_STATE)
print("Valid 비율:", VALID_SIZE)
print("CV fold 수:", CV_SPLITS)


프로젝트 경로: /Users/sonsugyung/skala/11th_week/data-mini-prj
커널 실행 파일: /Users/sonsugyung/skala/11th_week/data-mini-prj/.venv/bin/python
random_state: 42
Valid 비율: 0.2
CV fold 수: 5


## 01. 데이터 및 피처 준비
Batch 1 로딩  
Summary 및 Cycle 10·100 Qdlin 추출  
두 피처와 Target 생성

In [2]:
# Batch 1 로딩
print("Batch 1 로딩 중...")
mat = load_mat(BATCH_1_PATH)
batch = to_list_of_dicts(mat["batch"])

# 전처리
summary_df = extract_summary(batch)
qd10_df = extract_cycle_qdlin(batch, 10)
qd100_df = extract_cycle_qdlin(batch, 100)

# 피처 표 생성
feature_table = build_feature_table(
    summary_df,
    qd10_df,
    qd100_df,
)

print("셀 개수:", len(batch))
print("피처 표 크기:", feature_table.shape)
display(feature_table.head())

ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:ro

Batch 1 로딩 중...
셀 개수: 46
피처 표 크기: (46, 4)


,cell_id,log_delta_q_variance,mean_chargetime,cycle_life
0,0,-5.014427,13.250388,1190.0
1,1,-5.013525,13.240949,1179.0
2,2,-4.736565,13.245097,1177.0
3,3,-4.442179,11.932744,1226.0
4,4,-4.647309,11.932781,1227.0


## 02. Train/Valid 분할

입력 : log_delta_q_variance, mean_chargetime  
Target : cycle_life  
셀 단위 Hold-out 분할 : Train 80%, Valid 20%  
random_state : 42

In [3]:
# 피처, Target 분리
X, y = split_features_target(feature_table)

# Hold-out 분할
X_train, X_valid, y_train, y_valid = split_train_valid(X, y)

# 셀 중복 검사
assert set(X_train.index).isdisjoint(X_valid.index)
assert len(X_train) + len(X_valid) == len(X)

print("입력 피처:", X.columns.tolist())
print("Train 셀 수:", len(X_train))
print("Valid 셀 수:", len(X_valid))
print("Train/Valid 중복: 없음")

입력 피처: ['log_delta_q_variance', 'mean_chargetime']
Train 셀 수: 36
Valid 셀 수: 10
Train/Valid 중복: 없음


## 03. Train 교차검증 평가
Nested CV : 바깥 5-fold 평가, 안쪽 5-fold alpha 선택  
각 학습 fold 내부에서 StandardScaler, Ridge 학습  
Hold-out Valid 제외  
지표 - 셀별 검증 예측을 모은 MAPE(%)

In [4]:
# Train Nested CV
cv_predictions, train_cv_mape = evaluate_train_cv(
    X_train,
    y_train,
)

# 결과 확인
print(f"Train CV MAPE: {train_cv_mape:.2f}%")

cv_result = pd.DataFrame({
    "actual_cycle_life": y_train,
    "predicted_cycle_life": cv_predictions,
})

display(cv_result.head())

Train CV MAPE: 9.61%


,actual_cycle_life,predicted_cycle_life
6,636.0,782.701588
24,1017.0,833.028497
32,731.0,713.336738
19,788.0,716.410406
17,857.0,863.143896


## 04. alpha 선택 및 모델 학습
Train 내부 5-fold CV로 alpha 선택  
기준 : 평균 검증 MAPE 최소  
선택한 alpha로 Train 전체 재학습  
Hold-out Valid 제외

In [5]:
# alpha 선택·Train 재학습
search = select_alpha(X_train, y_train)

# 최종 Train 모델
best_model = search.best_estimator_
best_alpha = search.best_params_["ridge__alpha"]

print("선택한 alpha:", best_alpha)
print("Pipeline:", best_model)

선택한 alpha: 0.0001
Pipeline: Pipeline(steps=[('scaler', StandardScaler()),
                ('ridge', Ridge(alpha=np.float64(0.0001)))])


## 05. Hold-out Valid 평가
Train에서 학습한 Pipeline으로 Valid 10개 예측  
Train의 표준화 기준 적용  
Valid 재학습 없음  
지표 - MAPE(%)

In [6]:
# Valid 예측, 평가
valid_predictions, valid_mape = evaluate_model(
    best_model,
    X_valid,
    y_valid,
)

print(f"Valid MAPE: {valid_mape:.2f}%")

# 셀별 오차
valid_result = pd.DataFrame({
    "actual_cycle_life": y_valid,
    "predicted_cycle_life": valid_predictions,
})

valid_result["error_cycles"] = (
    valid_result["predicted_cycle_life"]
    - valid_result["actual_cycle_life"]
)
valid_result["absolute_percentage_error"] = (
    valid_result["error_cycles"].abs()
    / valid_result["actual_cycle_life"]
    * 100
)

display(valid_result)

Valid MAPE: 7.46%


,actual_cycle_life,predicted_cycle_life,error_cycles,absolute_percentage_error
39,625.0,669.913983,44.913983,7.186237
25,854.0,879.732144,25.732144,3.013132
26,870.0,785.517153,-84.482847,9.710672
44,616.0,636.909549,20.909549,3.394407
35,703.0,777.052504,74.052504,10.533784
41,1051.0,896.835184,-154.164816,14.668394
4,1227.0,1123.827272,-103.172728,8.408535
12,902.0,915.321284,13.321284,1.476861
8,879.0,867.520506,-11.479494,1.305972
3,1226.0,1042.995360,-183.004640,14.926969


## 06. 학습, 검증 결과 요약
Train CV : Nested CV 평가  
Valid : Hold-out 10개 셀 평가  
Train-Valid Gap : Valid MAPE − Train CV MAPE  
Gap 단위 : %p  
단일 Hold-out 결과의 표본 변동 고려

## 07. 최종 모델 학습
선택한 alpha 고정 : 0.0001  
Batch 1 전체 46개 셀 사용 (이전엔 36개 셀만 사용)  
StandardScaler, Ridge 재학습  
Batch 2 테스트용 모델 생성

In [7]:
# alpha 고정
final_model = build_pipeline()
final_model.set_params(ridge__alpha=best_alpha)

# Batch 1 전체 재학습
final_model.fit(X, y)

print("최종 학습 셀 수:", len(X))
print("최종 alpha:", final_model.named_steps["ridge"].alpha)
print("표준화 학습 셀 수:", final_model.named_steps["scaler"].n_samples_seen_)

최종 학습 셀 수: 46
최종 alpha: 0.0001
표준화 학습 셀 수: 46.0


## 08. 테스트 데이터 준비
평가 대상 - Batch 2  
학습 데이터와 동일한 전처리, 피처 계산  
Target 결측 셀 제외: 기존 EDA 기준  
테스트 데이터로 스케일러, 모델 학습 없음

In [8]:
# Batch 2 로딩
print("Batch 2 로딩 중...")
test_mat = load_mat(BATCH_2_PATH)
test_batch = to_list_of_dicts(test_mat["batch"])

# 전처리
test_summary_df = extract_summary(test_batch)
test_qd10_df = extract_cycle_qdlin(test_batch, 10)
test_qd100_df = extract_cycle_qdlin(test_batch, 100)

# 테스트 피처
test_feature_table = build_feature_table(
    test_summary_df,
    test_qd10_df,
    test_qd100_df,
)

# 피처·Target 분리
X_test, y_test = split_features_target(test_feature_table)

print("원본 셀 수:", len(test_batch))
print("평가 대상 셀 수:", len(test_feature_table))
print("테스트 피처 표 크기:", test_feature_table.shape)
display(test_feature_table.head())

ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:ro

Batch 2 로딩 중...


ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:ro

원본 셀 수: 47
평가 대상 셀 수: 39
테스트 피처 표 크기: (39, 4)


,cell_id,log_delta_q_variance,mean_chargetime,cycle_life
0,0,-3.486830,10.173261,477.0
1,1,-3.297617,10.199074,491.0
2,2,-3.459814,10.175028,424.0
3,3,-3.495808,10.173197,499.0
4,4,-3.232072,10.231934,444.0


## 09. 최종 테스트 평가
Batch 1 전체로 학습한 최종 Pipeline 사용  
Batch 2 : 39개 셀  
학습된 표준화 기준 적용  
지표 - MAPE(%)

In [9]:
# Test 예측, 평가
test_predictions, test_mape = evaluate_model(
    final_model,
    X_test,
    y_test,
)

print(f"Test MAPE: {test_mape:.2f}%")

# 셀별 오차
test_result = pd.DataFrame({
    "cell_id": test_feature_table["cell_id"],
    "actual_cycle_life": y_test,
    "predicted_cycle_life": test_predictions,
})

test_result["error_cycles"] = (
    test_result["predicted_cycle_life"]
    - test_result["actual_cycle_life"]
)
test_result["absolute_percentage_error"] = (
    test_result["error_cycles"].abs()
    / test_result["actual_cycle_life"]
    * 100
)

display(test_result.head())

Test MAPE: 29.43%


,cell_id,actual_cycle_life,predicted_cycle_life,error_cycles,absolute_percentage_error
0,0,477.0,656.001510,179.001510,37.526522
1,1,491.0,573.591271,82.591271,16.821033
2,2,424.0,644.237021,220.237021,51.942694
3,3,499.0,659.910508,160.910508,32.246595
4,4,444.0,545.015516,101.015516,22.751242


## 10. 성능 및 GAP 정리
Train CV: 9.61%  
Valid: 7.46%  
Test: 29.43%  
Gap : 뒤쪽 평가 오차 − 앞쪽 평가 오차  
Gap 단위: %p  
Valid : Train 36개로 학습한 모델  
Test : Batch 1 전체 46개로 재학습한 모델  

In [10]:
# 원논문 비교 목표
target_mape = 9.1

# 성능, Gap 표
performance_report = pd.DataFrame({
    "구분": [
        "Train (Batch 1 CV)",
        "Valid (Batch 1 Hold-out)",
        "Test (Batch 2)",
        "Gap (Train-Valid)",
        "Gap (Valid-Test)",
        "Gap (Target-Test)",
    ],
    "MAPE (%)": [
        train_cv_mape,
        valid_mape,
        test_mape,
        valid_mape - train_cv_mape,
        test_mape - valid_mape,
        test_mape - target_mape,
    ],
    "비고": [
        "Train 내부 Nested CV",
        "Hold-out 10개 셀",
        "Batch 1 전체 재학습 후 평가",
        "(+) : 과적합 의심 / 단위: %p",
        "(+) : 배치간 일반화 저하 의심 / 단위: %p",
        "Target : 원논문 9.1% / 단위: %p",
    ],
})

display(performance_report.round({"MAPE (%)": 2}))

,구분,MAPE (%),비고
0,Train (Batch 1 CV),9.61,Train 내부 Nested CV
1,Valid (Batch 1 Hold-out),7.46,Hold-out 10개 셀
2,Test (Batch 2),29.43,Batch 1 전체 재학습 후 평가
3,Gap (Train-Valid),-2.15,(+) : 과적합 의심 / 단위: %p
4,Gap (Valid-Test),21.97,(+) : 배치간 일반화 저하 의심 / 단위: %p
5,Gap (Target-Test),20.33,Target : 원논문 9.1% / 단위: %p


## 11. 기준 모델 비교

- 기준 모델 : DummyRegressor, 학습 Target 평균 예측
- Train CV : Ridge 바깥 CV와 같은 5-fold 분할, fold별 학습 평균 사용
- Valid : Train 36개로 학습한 기준 모델 사용
- Test : Batch 1 전체 46개로 학습한 기준 모델 사용
- MAPE 감소량 : Dummy MAPE − Ridge MAPE (%p)
- 최종 테스트 이후 추가한 기준선 비교이며, 테스트 결과로 Ridge 설정을 변경하지 않음


In [11]:
# 기준 모델 CV
baseline_cv_predictions, baseline_cv_mape = evaluate_baseline_cv(
    X_train, y_train,
)

# Train 평균·Valid 평가
baseline_model = build_baseline()
baseline_model.fit(X_train, y_train)
baseline_valid_predictions, baseline_valid_mape = evaluate_model(
    baseline_model, X_valid, y_valid,
)

# Batch 1 전체 평균·Test 평가
final_baseline = build_baseline()
final_baseline.fit(X, y)
baseline_test_predictions, baseline_test_mape = evaluate_model(
    final_baseline, X_test, y_test,
)

# 같은 평가 대상 비교
baseline_comparison = pd.DataFrame({
    "평가": ["Train (Batch 1 CV)", "Valid (Batch 1 Hold-out)", "Test (Batch 2)"],
    "Dummy MAPE (%)": [
        baseline_cv_mape, baseline_valid_mape, baseline_test_mape,
    ],
    "Ridge MAPE (%)": [
        train_cv_mape, valid_mape, test_mape,
    ],
})
baseline_comparison["MAPE 감소량 (%p)"] = (
    baseline_comparison["Dummy MAPE (%)"]
    - baseline_comparison["Ridge MAPE (%)"]
)

print("Train 기준 평균 :", float(y_train.mean()))
print("최종 기준 평균 :", float(y.mean()))
display(baseline_comparison.round(2))


Train 기준 평균 : 830.6666666666666
최종 기준 평균 : 844.7173913043479


,평가,Dummy MAPE (%),Ridge MAPE (%),MAPE 감소량 (%p)
0,Train (Batch 1 CV),18.63,9.61,9.02
1,Valid (Batch 1 Hold-out),19.21,7.46,11.75
2,Test (Batch 2),70.30,29.43,40.87


### 기준 모델 비교 해석

- Ridge : 모든 평가에서 DummyRegressor보다 낮은 MAPE
- 두 피처 활용 : 평균 예측 기준선 대비 예측 오차 감소
- Batch 2 : 기준선 대비 개선됐지만 MAPE 29.43%로 외부 일반화 한계 존재
- 원논문 목표 9.1% 대비 : +20.33%p
- Dummy의 높은 Test 오차 : Batch 1 평균 수명으로 Batch 2의 짧은 수명을 예측하는 한계
- 테스트 결과로 피처·모델 설정 변경 없음

## 12. 오류 분석
큰 오차 기준 : 절대 사이클 오차 상위 5개 셀  
상대 오차 기준 : 백분율 오차 상위 5개 셀  
비교 항목 : 예측 방향, 실제 수명, 입력 피처, 충전 정책

In [12]:
# 셀별 충전 정책
test_policies = (
    test_summary_df[["cell_id", "charging_policy"]]
    .drop_duplicates()
)

# 피처·정책 연결
error_analysis = (
    test_result.merge(
        test_feature_table[
            ["cell_id", "log_delta_q_variance", "mean_chargetime"]
        ],
        on="cell_id",
        validate="one_to_one",
    )
    .merge(
        test_policies,
        on="cell_id",
        validate="one_to_one",
    )
)

# 절대 오차·예측 방향
error_analysis["absolute_error_cycles"] = (
    error_analysis["error_cycles"].abs()
)
error_analysis["prediction_direction"] = (
    error_analysis["error_cycles"].map(
        lambda value: "과대예측" if value > 0
        else "과소예측" if value < 0
        else "일치"
    )
)

# 절대 오차 상위 5개
print("절대 사이클 오차 상위 5개")
display(
    error_analysis.nlargest(5, "absolute_error_cycles")
)

# 상대 오차 상위 5개
print("백분율 오차 상위 5개")
display(
    error_analysis.nlargest(5, "absolute_percentage_error")
)

절대 사이클 오차 상위 5개


,cell_id,actual_cycle_life,predicted_cycle_life,error_cycles,absolute_percentage_error,log_delta_q_variance,mean_chargetime,charging_policy,absolute_error_cycles,prediction_direction
18,18,449.0,751.800394,302.800394,67.438840,-3.706868,10.174147,5.2C(50%)-4.25C,302.800394,과대예측
6,6,393.0,674.617645,281.617645,71.658434,-3.529605,10.179292,3.6C(9%)-5C,281.617645,과대예측
15,15,396.0,668.622144,272.622144,68.843976,-3.515817,10.173059,3.6C(9%)-5C,272.622144,과대예측
9,9,791.0,1029.349826,238.349826,30.132721,-4.343991,10.039939,5.6C(26%)-4.5C-newstructure,238.349826,과대예측
2,2,424.0,644.237021,220.237021,51.942694,-3.459814,10.175028,5.2C(50%)-4.25C,220.237021,과대예측


백분율 오차 상위 5개


,cell_id,actual_cycle_life,predicted_cycle_life,error_cycles,absolute_percentage_error,log_delta_q_variance,mean_chargetime,charging_policy,absolute_error_cycles,prediction_direction
6,6,393.0,674.617645,281.617645,71.658434,-3.529605,10.179292,3.6C(9%)-5C,281.617645,과대예측
15,15,396.0,668.622144,272.622144,68.843976,-3.515817,10.173059,3.6C(9%)-5C,272.622144,과대예측
18,18,449.0,751.800394,302.800394,67.438840,-3.706868,10.174147,5.2C(50%)-4.25C,302.800394,과대예측
2,2,424.0,644.237021,220.237021,51.942694,-3.459814,10.175028,5.2C(50%)-4.25C,220.237021,과대예측
27,29,452.0,665.137475,213.137475,47.154309,-3.507814,10.173062,5.2C(58%)-4C,213.137475,과대예측
